# 01 — Data profiling

Inspect every supplied file before modeling. The package README warns the data are intentionally dirty.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
GOLDEN = ROOT / 'data' / 'golden'
CLEAN = ROOT / 'data' / 'clean'
plt.rcParams['figure.figsize'] = (9, 4)
assert (GOLDEN / 'metrics_monthly.parquet').exists(), 'Run python scripts/run_pipeline.py first'


In [ ]:

tables = sorted(p.stem for p in RAW.glob('*.csv') if p.stem != 'data_dictionary')
rows = []
for t in tables:
    df = pd.read_csv(RAW / f'{t}.csv')
    rows.append({
        'table': t, 'rows': len(df), 'cols': df.shape[1],
        'exact_dups': int(df.duplicated().sum()),
        'columns': ', '.join(df.columns),
    })
inv = pd.DataFrame(rows)
inv


## Date ranges and identifier uniqueness

In [ ]:

for t, col in [('payments','event_at'),('calls','event_at'),('daily_targeting','target_date'),('accounts','opened_at')]:
    s = pd.to_datetime(pd.read_csv(RAW / f'{t}.csv', usecols=[col])[col], errors='coerce')
    print(t, s.min(), '→', s.max(), 'n=', s.notna().sum())
print('accounts PK unique', pd.read_csv(RAW/'accounts.csv')['account_id'].is_unique)
print('payment_id unique?', pd.read_csv(RAW/'payments.csv')['payment_id'].is_unique)
print('agent_id unique in agents?', pd.read_csv(RAW/'agents.csv')['agent_id'].is_unique)


See `docs/data_inventory.md` for the full column-level inventory. Do not treat `data_dictionary.csv` as keys or metric logic — it is dtypes only.